# Fine-tuning Qwen2.5-Coder-1.5B for Kernel Log Classification

In this notebook, we showcase how to fine-tune the **Qwen2.5-Coder-1.5B** model on AWS Trainium for **kernel log anomaly detection**.

## Task Overview
The goal is to classify Linux kernel logs (system call sequences) as **normal** or **abnormal** (attack). This is a binary classification task using the DongTing dataset.

## Dataset: DongTing
DongTing is the first large-scale dataset dedicated to Linux kernel anomaly detection:
- **18,966 well-labeled sequences** (normal and attack)
- **Attack data**: 12,116 system call sequences from bug-triggering programs
- **Normal data**: 6,850 sequences from kernel regression test suites
- Data format: System call sequences (e.g., `read|write|open|close`)

## Workflow
1. Install dependencies
2. Prepare and load the DongTing dataset
3. Fine-tune Qwen2.5-Coder-1.5B with LoRA
4. Compile the model for Neuron
5. Run inference for kernel log classification

**Citation**: If you use DongTing, please cite:
```
Guoyun Duan, et al. "DongTing: A large-scale dataset for anomaly detection of the Linux kernel."
Journal of Systems and Software, 2023.
```

# 1. Install Requirements

Install Hugging Face Optimum Neuron and related libraries for training on AWS Trainium.

In [1]:
%cd /home/ubuntu/environment/FineTuning/HuggingFaceExample/01_finetuning/assets
%pip install -r requirements.txt
%pip install openpyxl pandas scikit-learn

/home/ubuntu/environment/FineTuning/HuggingFaceExample/01_finetuning/assets


/opt/aws_neuronx_venv_pytorch_latest/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Looking in indexes: https://pypi.org/simple, https://pip.repos.neuron.amazonaws.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pip.repos.neuron.amazonaws.com
Note: you may need to restart the kernel to use updated packages.


# 2. Data Preparation

## 2.1 Extract and Load the DongTing Dataset

The DongTing dataset contains:
- `Abnormal_data.zip`: Attack system call sequences
- `Normal_data.zip`: Normal system call sequences
- `Baseline.xlsx`: Metadata with labels and splits

We'll extract the data and prepare it in a format suitable for LLM fine-tuning.

In [2]:
import os
import zipfile
import pandas as pd
import json
from pathlib import Path
import random
from tqdm import tqdm

# Paths
data_dir = Path("/home/ubuntu/environment/Data/dataset")
output_dir = Path("/home/ubuntu/environment/ml/kernel_log_data")
output_dir.mkdir(parents=True, exist_ok=True)

# Extract data if not already extracted
abnormal_zip = data_dir / "Abnormal_data.zip"
normal_zip = data_dir / "Normal_data.zip"

abnormal_dir = output_dir / "Abnormal_data"
normal_dir = output_dir / "Normal_data"

# Helper to show a few entries inside a zip for debugging
def _print_zip_contents(zippath, max_entries=10):
    try:
        with zipfile.ZipFile(zippath, 'r') as zf:
            names = zf.namelist()[:max_entries]
            print(f"  {zippath.name} contains {len(zf.namelist())} entries. Sample: {names}")
    except Exception as e:
        print(f"  Could not read zip {zippath}: {e}")

if not abnormal_dir.exists() and abnormal_zip.exists():
    print("Extracting abnormal data...")
    _print_zip_contents(abnormal_zip)
    with zipfile.ZipFile(abnormal_zip, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print("Abnormal data extracted!")

if not normal_dir.exists() and normal_zip.exists():
    print("Extracting normal data...")
    _print_zip_contents(normal_zip)
    with zipfile.ZipFile(normal_zip, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    print("Normal data extracted!")

# After extraction, show a short directory listing to help debugging
print(f"Data directory: {output_dir}")
print(f"Abnormal data exists: {abnormal_dir.exists()}")
print(f"Normal data exists: {normal_dir.exists()}")
try:
    sample = list(output_dir.rglob('*'))[:20]
    print("Sample files under output_dir (up to 20):")
    for p in sample:
        print(f"  - {p.relative_to(output_dir)}")
except Exception as e:
    print(f"  Could not list output_dir: {e}")

Data directory: /home/ubuntu/environment/ml/kernel_log_data
Abnormal data exists: True
Normal data exists: True
Sample files under output_dir (up to 20):
  - Abnormal_data
  - train.jsonl
  - Normal_data
  - validation.jsonl
  - test.jsonl
  - prepared
  - Abnormal_data/kernel_v5140-186.zip
  - Abnormal_data/kernel_v580-313
  - Abnormal_data/kernel_v5140-186
  - Abnormal_data/kernel_v5130-177.zip
  - Abnormal_data/kernel_v5110-536
  - Abnormal_data/kernel_v580-313.zip
  - Abnormal_data/kernel_v5150-90.zip
  - Abnormal_data/kernel_v4150-1540
  - Abnormal_data/kernel_v5160-107.zip
  - Abnormal_data/kernel_v5130-177
  - Abnormal_data/kernel_v560-530
  - Abnormal_data/kernel_v5120-397
  - Abnormal_data/kernel_v4140-184.zip
  - Abnormal_data/kernel_v4190-821


## 2.2 Load Baseline Metadata

The Baseline.xlsx file contains information about train/validation/test splits.

In [3]:
# Load baseline metadata if available
baseline_path = data_dir / "Baseline.xlsx"

if baseline_path.exists():
    baseline_df = pd.read_excel(baseline_path)
    print(f"Baseline metadata loaded: {len(baseline_df)} entries")
    print(f"Columns: {baseline_df.columns.tolist()}")
    print("\nFirst few rows:")
    print(baseline_df.head())
else:
    print("Baseline.xlsx not found. Will create splits manually.")
    baseline_df = None

Baseline metadata loaded: 18966 entries
Columns: ['_id', 'kcb_bl_time', 'kcb_bug_name', 'kcb_master_line_ver', 'kcb_seq_bug_id', 'kcb_seq_class', 'kcb_seq_id', 'kcb_seq_lables', 'kcb_seq_poc_id', 'kcb_syscall_counts', 'kcb_syscall_sizes']

First few rows:
                        _id               kcb_bl_time  \
0  628e76675ad37931ac209a05  Thu May 26 02:33:11 2022   
1  628e76675ad37931ac209a06  Thu May 26 02:33:11 2022   
2  628e76675ad37931ac209a07  Thu May 26 02:33:11 2022   
3  628e76675ad37931ac209a08  Thu May 26 02:33:11 2022   
4  628e76675ad37931ac209a09  Thu May 26 02:33:11 2022   

                                        kcb_bug_name  kcb_master_line_ver  \
0  general_protection_fault_in___pm_runtime_resum...                  5.2   
1  general_protection_fault_in___pm_runtime_resum...                  5.2   
2  general_protection_fault_in___pm_runtime_resum...                  5.3   
3  general_protection_fault_in___pm_runtime_resum...                  5.3   
4  general_prote

## 2.3 Process System Call Sequences into Training Format

We'll convert the system call sequences into a conversational format suitable for instruction fine-tuning.

**Format**:
```
System: You are a kernel log classifier...
User: Classify this kernel log: <syscalls>
Assistant: normal / abnormal
```

In [4]:
from tqdm import tqdm
import concurrent.futures
import os
import zipfile
from pathlib import Path

def read_syscall_file(filepath):
    """Read system call sequence from file."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read().strip()
            # System calls are typically separated by pipes or spaces
            if '|' in content:
                syscalls = content.split('|')
            else:
                syscalls = content.split()
            return ' '.join(syscalls[:512])  # Limit to first 512 syscalls
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None


def create_training_example(syscall_sequence, label):
    """Create a training example in chat format."""
    system_prompt = """You are a Linux kernel security expert specialized in anomaly detection. Your task is to analyze system call sequences and classify them as either 'normal' (benign) or 'abnormal' (potential attack or bug)."""

    user_message = f"""Analyze the following kernel system call sequence and classify it as normal or abnormal:

{syscall_sequence}"""

    assistant_message = label

    # Format for Qwen chat template
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_message}
    ]

    return conversation


# Helper: extract nested zip files (the dataset packs inner zip archives)
def extract_nested_zips(root_dir):
    """Recursively extract .zip files found under root_dir in-place."""
    root = Path(root_dir)
    zip_files = list(root.rglob('*.zip'))
    if not zip_files:
        return 0

    # Parallel extraction helper (IO-bound -> threads are fine)
    def _extract_one(zpath):
        try:
            p = Path(zpath)
            dest = p.with_suffix('')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(p, 'r') as zf:
                zf.extractall(dest)
            return True
        except Exception as e:
            print(f"Failed to extract {zpath}: {e}")
            return False

    # use a thread pool to extract multiple archives in parallel
    max_workers = min(8, (os.cpu_count() or 2) * 2)
    count = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        for success in tqdm(ex.map(_extract_one, [str(z) for z in zip_files]), total=len(zip_files), desc="Extracting nested zips"):
            if success:
                count += 1
    return count


# Process files
def process_data_files(data_dir, label, max_samples=None):
    """Process all files in a directory (handles nested zip extractions)."""
    examples = []
    # Ensure nested zip archives are extracted (many inner zips contain the actual .txt sequences)
    extracted = extract_nested_zips(data_dir)
    if extracted:
        print(f"Extracted {extracted} nested zip archives under {data_dir}")
    # Now search for .txt files
    files = list(Path(data_dir).rglob('*.txt'))
    if max_samples:
        files = files[:max_samples]
    print(f"Processing {len(files)} files for label '{label}'...")
    for filepath in tqdm(files, desc=f"Processing {label}"):
        syscalls = read_syscall_file(filepath)
        if syscalls:
            example = create_training_example(syscalls, label)
            examples.append(example)
    return examples


# Process both normal and abnormal data (this cell orchestrates preparation + processing)
print("Processing dataset...")
all_examples = []

# --- New: prepare .log -> .txt step with a fast-sample mode to avoid long runs ---

def _make_flat_name(path: Path):
    """Create a safe flattened filename from a path by joining parts with underscores."""
    parts = list(path.parts)
    name = "_".join(parts)
    # remove any leading slashes
    name = name.lstrip("._/")
    return name


def prepare_logs_to_txt(root_dir, out_dir, max_tokens=512, sample_limit=None, flatten=False):
    """Prepare .log files into trimmed .txt syscall-like files.

    If sample_limit is set, only the first N .log files (sorted) will be processed. This speeds up
    local experimentation and avoids hours-long preprocessing.

    If flatten=True, all output files will be written directly under `out_dir` with unique
    flattened filenames (path components joined with underscores). Otherwise, the relative
    directory structure under `root_dir` is preserved.
    """
    root = Path(root_dir)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    log_files = sorted(list(root.rglob('*.log')))
    print(f'Found {len(log_files)} .log files under {root_dir}')
    if sample_limit is not None and sample_limit > 0:
        log_files = log_files[:sample_limit]
        print(f'Chose sample_limit={sample_limit}, processing {len(log_files)} files')

    for p in tqdm(log_files, desc='Preparing log files'):
        try:
            text = p.read_text(encoding='utf-8', errors='ignore')
            # heuristics: prefer lines containing '|' (pipe-separated syscalls)
            lines = [l.strip() for l in text.splitlines() if l.strip()]
            candidate = None
            for l in lines:
                if '|' in l and len(l.split('|')) > 2:
                    candidate = l
                    break
            if candidate is None:
                # fallback: join lines, replace newlines with space and collapse whitespace
                candidate = ' '.join(lines)
            tokens = candidate.split()
            trimmed = ' '.join(tokens[:max_tokens])

            if flatten:
                flat_name = _make_flat_name(p.relative_to(root))
                out_path = out / (flat_name + '.txt')
            else:
                try:
                    rel = p.relative_to(root)
                except Exception:
                    rel = Path(p.name)
                out_path = out / rel.with_suffix('.txt')
                out_path.parent.mkdir(parents=True, exist_ok=True)

            # write the prepared text
            out_path.write_text(trimmed, encoding='utf-8')
        except Exception as e:
            print(f"Failed to prepare {p}: {e}")
    print(f'Prepared files in {out}')


# Run prepare step for abnormal and normal directories into a prepared folder
prepared_root = output_dir / 'prepared'
# If this is a quick run (to avoid hours of preprocessing), set SAMPLE_PER_CLASS to a small number
SAMPLE_PER_CLASS = int(os.environ.get('SAMPLE_PER_CLASS', '100'))  # default 100 files per class
# Flatten outputs by default to speed up downstream path handling
if abnormal_dir.exists():
    prepare_logs_to_txt(abnormal_dir, prepared_root / 'Abnormal_data', sample_limit=SAMPLE_PER_CLASS, flatten=True)
if normal_dir.exists():
    prepare_logs_to_txt(normal_dir, prepared_root / 'Normal_data', sample_limit=SAMPLE_PER_CLASS, flatten=True)
# After preparing, update paths used by process_data_files to point to prepared_root
abnormal_dir = prepared_root / 'Abnormal_data'
normal_dir = prepared_root / 'Normal_data'
print('Preparation complete — now run the splitting cell to continue')


# Now process prepared files into training examples
if abnormal_dir.exists():
    abnormal_examples = process_data_files(abnormal_dir, "abnormal", max_samples=5000)
    all_examples.extend(abnormal_examples)
    print(f"Abnormal examples: {len(abnormal_examples)}")

if normal_dir.exists():
    normal_examples = process_data_files(normal_dir, "normal", max_samples=5000)
    all_examples.extend(normal_examples)
    print(f"Normal examples: {len(normal_examples)}")

print(f"\nTotal examples: {len(all_examples)}")

# Shuffle the data
import random
random.seed(42)
random.shuffle(all_examples)


Processing dataset...
Found 12116 .log files under /home/ubuntu/environment/ml/kernel_log_data/Abnormal_data
Chose sample_limit=100, processing 100 files


Preparing log files:   0%|          | 0/100 [00:00<?, ?it/s]

Preparing log files: 100%|██████████| 100/100 [00:01<00:00, 77.79it/s]


Prepared files in /home/ubuntu/environment/ml/kernel_log_data/prepared/Abnormal_data
Found 6850 .log files under /home/ubuntu/environment/ml/kernel_log_data/Normal_data
Chose sample_limit=100, processing 100 files


Preparing log files: 100%|██████████| 100/100 [00:00<00:00, 1425.62it/s]


Prepared files in /home/ubuntu/environment/ml/kernel_log_data/prepared/Normal_data
Preparation complete — now run the splitting cell to continue
Processing 1481 files for label 'abnormal'...


Processing abnormal: 100%|██████████| 1481/1481 [01:32<00:00, 16.08it/s]  


Abnormal examples: 1481
Processing 100 files for label 'normal'...


Processing normal: 100%|██████████| 100/100 [00:00<00:00, 33112.05it/s]

Normal examples: 100

Total examples: 1581


## 2.4 Create Train/Validation/Test Splits

In [5]:

# Next step: prepare extracted .log files into .txt syscall-like files for training
def prepare_logs_to_txt(root_dir, out_dir, max_tokens=512):
    root = Path(root_dir)
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    log_files = list(root.rglob('*.log'))
    print(f'Found {len(log_files)} .log files under {root_dir}')
    for p in tqdm(log_files, desc='Preparing log files'):
        try:
            rel = p.relative_to(root)
        except Exception:
            rel = p.name
        out_path = out / rel.with_suffix('.txt') if isinstance(rel, Path) else out / (p.name + '.txt')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        text = p.read_text(errors='ignore')
        # heuristics: prefer lines containing '|' (pipe-separated syscalls)
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        candidate = None
        for l in lines:
            if '|' in l and len(l.split('|'))>2:
                candidate = l
                break
        if candidate is None:
            # fallback: join lines, replace newlines with space and collapse whitespace
            candidate = ' '.join(lines)
        tokens = candidate.split()
        trimmed = ' '.join(tokens[:max_tokens])
        out_path.write_text(trimmed, encoding='utf-8')
    print(f'Prepared files in {out}')

# Run prepare step for abnormal and normal directories into a prepared folder
prepared_root = output_dir / 'prepared'
if abnormal_dir.exists():
    prepare_logs_to_txt(abnormal_dir, prepared_root / 'Abnormal_data')
if normal_dir.exists():
    prepare_logs_to_txt(normal_dir, prepared_root / 'Normal_data')
# After preparing, update paths used by process_data_files to point to prepared_root
abnormal_dir = prepared_root / 'Abnormal_data'
normal_dir = prepared_root / 'Normal_data'
print('Preparation complete — now run the splitting cell to continue')

Found 0 .log files under /home/ubuntu/environment/ml/kernel_log_data/prepared/Abnormal_data


Preparing log files: 0it [00:00, ?it/s]


Prepared files in /home/ubuntu/environment/ml/kernel_log_data/prepared/Abnormal_data
Found 0 .log files under /home/ubuntu/environment/ml/kernel_log_data/prepared/Normal_data


Preparing log files: 0it [00:00, ?it/s]

Prepared files in /home/ubuntu/environment/ml/kernel_log_data/prepared/Normal_data
Preparation complete — now run the splitting cell to continue


In [6]:
# Split data: 80% train, 10% validation, 10% test
n_train = int(0.8 * len(all_examples))
n_val = int(0.1 * len(all_examples))

train_data = all_examples[:n_train]
val_data = all_examples[n_train:n_train + n_val]
test_data = all_examples[n_train + n_val:]

print(f"Train: {len(train_data)} examples")
print(f"Validation: {len(val_data)} examples")
print(f"Test: {len(test_data)} examples")

# Save to JSONL format
def save_jsonl(data, filepath):
    with open(filepath, 'w') as f:
        for item in data:
            f.write(json.dumps({"messages": item}) + '\n')

train_file = output_dir / "train.jsonl"
val_file = output_dir / "validation.jsonl"
test_file = output_dir / "test.jsonl"

save_jsonl(train_data, train_file)
save_jsonl(val_data, val_file)
save_jsonl(test_data, test_file)

print(f"\nData saved to:")
print(f"  Train: {train_file}")
print(f"  Validation: {val_file}")
print(f"  Test: {test_file}")

# Show example
print("\n=== Example Training Instance ===")
print(json.dumps(train_data[0], indent=2))

Train: 1264 examples
Validation: 158 examples
Test: 159 examples

Data saved to:
  Train: /home/ubuntu/environment/ml/kernel_log_data/train.jsonl
  Validation: /home/ubuntu/environment/ml/kernel_log_data/validation.jsonl
  Test: /home/ubuntu/environment/ml/kernel_log_data/test.jsonl

=== Example Training Instance ===
[
  {
    "role": "system",
    "content": "You are a Linux kernel security expert specialized in anomaly detection. Your task is to analyze system call sequences and classify them as either 'normal' (benign) or 'abnormal' (potential attack or bug)."
  },
  {
    "role": "user",
    "content": "Analyze the following kernel system call sequence and classify it as normal or abnormal:\n\nexecve brk arch_prctl access openat newfstatat mmap close openat read pread64 pread64 pread64 newfstatat mmap pread64 mmap mmap mmap mmap mmap close mmap arch_prctl mprotect mprotect mprotect munmap mmap mmap mmap openat exit_group"
  },
  {
    "role": "assistant",
    "content": "abnormal"


# 3. Create Fine-tuning Script

We'll create a custom training script for kernel log classification.

# 4. Fine-tuning

Now we'll fine-tune Qwen3-1.7B on the kernel log classification task using Neuron-optimized training.

## Training Parameters:
- **Model**: Qwen/Qwen3-1.7B
- **Technique**: LoRA (Low-Rank Adaptation)
- **LoRA rank**: 16
- **LoRA alpha**: 32
- **Learning rate**: 5e-5
- **Max steps**: 1000
- **Batch size**: 2 per device
- **Tensor parallelism**: 2 NeuronCores

## Key Changes for Neuron Support:
The training script now uses `NeuronModelForCausalLM` and `NeuronSFTTrainer` from `optimum-neuron`, which properly support tensor parallelism on AWS Trainium. This is the same approach used in the working SQL example notebook.

In [ ]:
# Full fine-tuning run using shell magic for live streaming output
import os
from pathlib import Path

# Set cache environment variables to reuse compiled artifacts
%env XDG_CACHE_HOME=/home/ubuntu/.cache
%env OPTIMUM_NEURON_CACHE=/home/ubuntu/.cache/huggingface/hub/models--aws-neuron--optimum-neuron-cache
%env NEURON_COMPILE_CACHE=/var/tmp/neuron-compile-cache
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1

OUTPUT_DIR = Path.home() / 'environment' / 'ml' / 'qwen_kernel_log'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')
print('Environment variables set for cache reuse')

# Run torchrun with shell magic - output streams directly to the notebook
!torchrun \
    --nnodes 1 \
    --nproc_per_node 2 \
    /home/ubuntu/environment/FineTuning/HuggingFaceExample/01_finetuning/assets/finetune_kernel_log.py \
    --model_id Qwen/Qwen3-1.7B \
    --tokenizer_id Qwen/Qwen3-1.7B \
    --train_data /home/ubuntu/environment/ml/kernel_log_data/train.jsonl \
    --val_data /home/ubuntu/environment/ml/kernel_log_data/validation.jsonl \
    --output_dir {OUTPUT_DIR} \
    --bf16 True \
    --gradient_checkpointing True \
    --gradient_accumulation_steps 1 \
    --learning_rate 5e-5 \
    --max_steps 1000 \
    --per_device_train_batch_size 2 \
    --tensor_parallel_size 2 \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_dropout 0.05 \
    --dataloader_drop_last True \
    --disable_tqdm False \
    --logging_steps 10 \
    --save_steps 500

env: XDG_CACHE_HOME=/home/ubuntu/.cache
env: OPTIMUM_NEURON_CACHE=/home/ubuntu/.cache/huggingface/hub/models--aws-neuron--optimum-neuron-cache
env: NEURON_COMPILE_CACHE=/var/tmp/neuron-compile-cache
env: OMP_NUM_THREADS=1
env: MKL_NUM_THREADS=1
Output directory: /home/ubuntu/environment/ml/qwen_kernel_log
Environment variables set for cache reuse
/opt/aws_neuronx_venv_pytorch_2_7_nxd_inference/lib/python3.10/site-packages/neuronx_distributed/parallel_layers/layers.py:14: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_7_nxd_inference/lib/python3.10/site-packages/neuronx_distributed/parallel_layers/layers.py:14: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_7_nxd_inference/lib/python3.10/site-packages/neuronx_distributed/parallel_layers/layers.py:14: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use n

# 5. Model Compilation for Neuron

After fine-tuning, we compile the model for optimized inference on AWS Trainium/Inferentia.

In [ ]:
!optimum-cli export neuron \
  --model /home/ubuntu/environment/ml/qwen_kernel_log/merged_model \
  --task text-generation \
  --sequence_length 1024 \
  --batch_size 1 \
  /home/ubuntu/environment/ml/qwen_kernel_log/compiled_model

# 6. Inference - Kernel Log Classification

Now we'll test the fine-tuned model on kernel log classification tasks.

In [ ]:
%pip install optimum-neuron[vllm]

In [ ]:
import os
import json
from vllm import LLM, SamplingParams

# Load the compiled model
llm = LLM(
    model="/home/ubuntu/environment/ml/qwen_kernel_log/compiled_model",
    max_num_seqs=1,
    max_model_len=1024,
    device="neuron",
    tensor_parallel_size=2,
    override_neuron_config={}
)

# Load test examples
test_file = "/home/ubuntu/environment/ml/kernel_log_data/test.jsonl"
test_examples = []
with open(test_file, 'r') as f:
    for line in f:
        test_examples.append(json.loads(line))

# Select a few test cases
sample_tests = test_examples[:5]

# Create prompts
prompts = []
for example in sample_tests:
    messages = example["messages"]
    # Format without the assistant's response for inference
    prompt_messages = messages[:-1]  # Exclude assistant message
    
    # Manually format the prompt
    prompt = f"""<|im_start|>system
{messages[0]['content']}<|im_end|>
<|im_start|>user
{messages[1]['content']}<|im_end|>
<|im_start|>assistant
"""
    prompts.append((prompt, messages[2]['content']))  # Store true label

# Configure sampling
sampling_params = SamplingParams(
    max_tokens=50,
    temperature=0.1,  # Low temperature for classification
    top_p=0.9
)

# Run inference
print("=" * 80)
print("KERNEL LOG CLASSIFICATION RESULTS")
print("=" * 80)

outputs = llm.generate([p[0] for p in prompts], sampling_params)

correct = 0
for i, output in enumerate(outputs):
    predicted = output.outputs[0].text.strip().lower()
    true_label = prompts[i][1].lower()
    
    # Check if prediction contains the correct label
    is_correct = true_label in predicted
    if is_correct:
        correct += 1
    
    print(f"\n--- Test Case {i+1} ---")
    print(f"True Label: {true_label}")
    print(f"Predicted: {predicted[:100]}...")
    print(f"Correct: {'✓' if is_correct else '✗'}")
    print("-" * 80)

accuracy = correct / len(outputs) * 100
print(f"\n{'=' * 80}")
print(f"Sample Accuracy: {accuracy:.1f}% ({correct}/{len(outputs)})")
print(f"{'=' * 80}")

# 7. Comprehensive Evaluation

Let's evaluate on a larger test set and compute metrics.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Evaluate on more examples (adjust based on your resources)
num_eval_samples = min(100, len(test_examples))
eval_examples = test_examples[:num_eval_samples]

prompts_eval = []
true_labels = []

for example in eval_examples:
    messages = example["messages"]
    prompt = f"""<|im_start|>system
{messages[0]['content']}<|im_end|>
<|im_start|>user
{messages[1]['content']}<|im_end|>
<|im_start|>assistant
"""
    prompts_eval.append(prompt)
    true_labels.append(messages[2]['content'].lower())

print(f"Evaluating on {len(prompts_eval)} examples...")

# Generate predictions in batches
batch_size = 10
all_predictions = []

for i in range(0, len(prompts_eval), batch_size):
    batch = prompts_eval[i:i+batch_size]
    outputs = llm.generate(batch, sampling_params)
    
    for output in outputs:
        pred_text = output.outputs[0].text.strip().lower()
        # Extract label
        if 'abnormal' in pred_text:
            pred_label = 'abnormal'
        elif 'normal' in pred_text:
            pred_label = 'normal'
        else:
            pred_label = 'unknown'
        all_predictions.append(pred_label)
    
    print(f"Processed {min(i+batch_size, len(prompts_eval))}/{len(prompts_eval)}")

# Calculate metrics
print("\n" + "=" * 80)
print("EVALUATION RESULTS")
print("=" * 80)

print("\nClassification Report:")
print(classification_report(true_labels, all_predictions, 
                          labels=['normal', 'abnormal'],
                          target_names=['Normal', 'Abnormal']))

print("\nConfusion Matrix:")
cm = confusion_matrix(true_labels, all_predictions, labels=['normal', 'abnormal'])
print("                Predicted")
print("              Normal  Abnormal")
print(f"True Normal    {cm[0][0]:6d}  {cm[0][1]:8d}")
print(f"     Abnormal  {cm[1][0]:6d}  {cm[1][1]:8d}")

accuracy = np.sum(np.array(true_labels) == np.array(all_predictions)) / len(true_labels) * 100
print(f"\nOverall Accuracy: {accuracy:.2f}%")

# 8. Conclusion

## Summary
You have successfully:
1. ✅ Prepared the DongTing kernel log dataset
2. ✅ Fine-tuned Qwen2.5-Coder-1.5B for binary classification (normal vs abnormal)
3. ✅ Compiled the model for AWS Neuron
4. ✅ Evaluated the model on kernel log classification

## Next Steps
- **Hyperparameter tuning**: Experiment with different learning rates, LoRA ranks, and training steps
- **Data augmentation**: Use more training data from the full DongTing dataset
- **Multi-class classification**: Extend to classify specific types of attacks
- **Deploy as API**: Create a REST API for real-time kernel log classification
- **Ensemble methods**: Combine with traditional ML models from DongTing benchmarks

## References
- DongTing Dataset: http://doi.org/10.5281/zenodo.6627050
- Paper: Guoyun Duan et al., "DongTing: A large-scale dataset for anomaly detection of the Linux kernel", JSS 2023
- Optimum Neuron: https://github.com/huggingface/optimum-neuron